# Лабораторная работа 1 - CV

Выполнил ст-т Даутов Т. Б. (М8О-406Б-22)

# 1. Выбор условий и метрик

a) Обоснование датасета: Медицинская сегментация опухолей мозга по МРТ критически важна для автоматизации диагностики, планирования хирургических вмешательств и отслеживания динамики лечения. Датасет содержит 4 класса, разные проекции (сагиттальная, аксиальная, корональная) и ручные аннотации, что делает его репрезентативным для прототипирования клинических систем.

Примечание: Датасет размечен в формате YOLO (bbox), но задача требует семантической сегментации. В коде реализована конвертация bounding box в пиксельные маски, что является стандартным шагом при работе с подобными наборами данных.

b) Метрики качества:
- Dice Coefficient (F1-score для сегментации): Устойчив к классовой несбалансированности, стандарт в медицинской визуализации.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q segmentation-models-pytorch albumentations timm opencv-python-headless xmltodict pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.5 MB/s eta 0:00:00


In [3]:
import os, cv2, glob, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss
import albumentations as A
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
def yolo_to_mask(label_path, img_shape, num_classes=4):
    """Конвертирует YOLO txt в многоклассовую маску (0-3)"""
    mask = np.zeros((img_shape[0], img_shape[1]), dtype=np.uint8)
    if not os.path.exists(label_path):
        return mask
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if not parts: continue
            cls_id = int(float(parts[0]))
            xc, yc, w, h = map(float, parts[1:])
            # Преобразование нормализованных координат в пиксели
            x1 = int((xc - w/2) * img_shape[1])
            y1 = int((yc - h/2) * img_shape[0])
            x2 = int((xc + w/2) * img_shape[1])
            y2 = int((yc + h/2) * img_shape[0])
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img_shape[1], x2), min(img_shape[0], y2)
            mask[y1:y2, x1:x2] = cls_id
    return mask

class BrainSegDataset(Dataset):
    def __init__(self, data_dir, img_size=(256, 256), transform=None):
        self.data_dir = data_dir
        self.img_size = img_size
        self.transform = transform
        self.image_paths, self.label_paths = [], []

        for cls in os.listdir(data_dir):
            cls_path = os.path.join(data_dir, cls)
            if not os.path.isdir(cls_path): continue
            img_f, lbl_f = os.path.join(cls_path, 'images'), os.path.join(cls_path, 'labels')
            if not (os.path.isdir(img_f) and os.path.isdir(lbl_f)): continue

            for fname in os.listdir(img_f):
                if fname.endswith(('.jpg', '.jpeg', '.png')):
                    self.image_paths.append(os.path.join(img_f, fname))
                    self.label_paths.append(os.path.join(lbl_f, fname.rsplit('.', 1)[0] + '.txt'))

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(self.image_paths[idx]), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        mask = yolo_to_mask(self.label_paths[idx], (h, w))

        img = cv2.resize(img, self.img_size)
        mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']

        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).long()
        return img, mask

DATA_DIR = '/content/drive/MyDrive/КиберфизЛР/'
IMG_SIZE = (256, 256)
NUM_CLASSES = 4
BATCH_SIZE = 8

# Разделение на train/val
full_dataset = BrainSegDataset(DATA_DIR, IMG_SIZE, transform=A.Compose([A.Normalize()]))
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [5]:
def compute_metrics(model, loader, device, num_classes):
    model.eval()
    dice_scores, iou_scores, acc_scores = [], [], []
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Evaluating"):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs).argmax(dim=1)

            for b in range(preds.size(0)):
                p, m = preds[b], masks[b]
                # Dice
                intersection = ((p == m) & (p > 0)).sum().float()
                dice = (2. * intersection) / ((p > 0).sum() + (m > 0).sum() + 1e-6)
                dice_scores.append(dice.item())

                # IoU
                union = ((p == m) | (p > 0) | (m > 0)).sum().float()
                iou = intersection / (union + 1e-6)
                iou_scores.append(iou.item())

                # Accuracy
                acc = (p == m).sum().float() / m.numel()
                acc_scores.append(acc.item())
    return np.mean(dice_scores), np.mean(iou_scores), np.mean(acc_scores)

def train_model(model, train_loader, val_loader, epochs, lr, loss_fn, optimizer, scheduler=None, desc="Baseline"):
    model.to(device)
    train_losses, val_dice = [], []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"{desc} Epoch {epoch+1}/{epochs}")
        for imgs, masks in pbar:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = loss_fn(outputs, masks)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        val_dice_score, _, _ = compute_metrics(model, val_loader, device, NUM_CLASSES)
        train_losses.append(running_loss / len(train_loader))
        val_dice.append(val_dice_score)
        if scheduler: scheduler.step()
        print(f"  Val Dice: {val_dice_score:.4f}")
    return model, train_losses, val_dice

# Бейзлайн (SMP CNN & Transformer)

In [ ]:
# 1. CNN Baseline (U-Net + ResNet34)
cnn_base = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", classes=NUM_CLASSES)
opt_cnn = optim.Adam(cnn_base.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
cnn_base, _, cnn_base_dice = train_model(cnn_base, train_loader, val_loader, epochs=10, lr=1e-3,
                                          loss_fn=loss_fn, optimizer=opt_cnn, desc="SMP CNN Base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Evaluating: 100%|██████████| 119/119 [04:04<00:00,  2.06s/it]


  Val Dice: 0.1694


...

  Val Dice: 0.4844


Evaluating: 100%|██████████| 119/119 [00:12<00:00,  9.37it/s]


  Val Dice: 0.5615


Evaluating: 100%|██████████| 119/119 [00:12<00:00,  9.31it/s]

  Val Dice: 0.5042


In [ ]:
# 2. Transformer Baseline (U-Net + ViT-tiny)
vit_base = smp.Unet(encoder_name="mit_b0", encoder_weights="imagenet", classes=NUM_CLASSES)
opt_vit = optim.Adam(vit_base.parameters(), lr=1e-4)
vit_base, _, vit_base_dice = train_model(vit_base, train_loader, val_loader, epochs=10, lr=1e-4,
                                          loss_fn=loss_fn, optimizer=opt_vit, desc="SMP ViT Base")

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

Evaluating: 100%|██████████| 119/119 [00:13<00:00,  8.62it/s]


  Val Dice: 0.0000


Evaluating: 100%|██████████| 119/119 [00:12<00:00,  9.65it/s]


...

  Val Dice: 0.5995


Evaluating: 100%|██████████| 119/119 [00:12<00:00,  9.45it/s]


  Val Dice: 0.6185


Evaluating: 100%|██████████| 119/119 [00:12<00:00,  9.54it/s]

  Val Dice: 0.6199


На базовой конфигурации трансформерная архитектура (MiT-B0 + U-Net) показала преимущество над свёрточной (ResNet34 + U-Net) по метрике Dice на 11.6%.
|Модель|Энкодер|Final Dice|Best Dice|Время эпохи|Наблюдения|
|-|-|-|-|-|-|
|SMP CNN Base|ResNet34|0.504|0.562 (ep9)|~67 сек|Стабильный рост, небольшой спад на ep10 (возможное переобучение)|
|SMP ViT Base|MiT-B0|0.620|0.620 (ep10)|~63 сек|Медленный старт (ep1-3), затем быстрый рост, лучшая финальная метрика|

Для медицинских изображений с глобальными структурными зависимостями (анатомия мозга) трансформерные энкодеры эффективнее свёрточных, но требуют больше эпох для выхода на плато.

Значение Dice около 0.5–0.6 для бейзлайна является ожидаемым результатом при обучении на bounding box (а не pixel-wise масках). Маска вносит шум, ограничивающий верхнюю границу качества.

# Улучшенный бейзлайн (Аугментации + AdamW + Cosine LR)

**Формулировка гипотез улучшения бейзлайна**

Для повышения качества сегментации были выдвинуты следующие гипотезы:
- Аугментации данных: Применение геометрических (отражения, повороты, масштабирование) и фотометрических (яркость, контраст) трансформаций расширит пространство признаков, повысит обобщающую способность моделей и снизит переобучение на ограниченном датасете.
- Оптимизатор и гиперпараметры: Замена Adam на AdamW с добавлением weight_decay и расписания CosineAnnealingLR обеспечит более стабильную сходимость, уменьшит колебания функции потерь и улучшит финальные метрики.
- Выбор архитектуры: Трансформерные энкодеры (MiT/SegFormer) должны эффективнее свёрточных (ResNet) выявлять глобальные структурные зависимости в МРТ-снимках, что повысит Dice и IoU.
- Предобучение: Инициализация энкодеров весами ImageNet даст значительный прирост качества по сравнению с обучением «с нуля» за счёт уже сформированных универсальных визуальных признаков.

In [7]:
# Улучшенный пайплайн аугментаций
strong_aug = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Normalize()
])

train_ds_aug = BrainSegDataset(DATA_DIR, IMG_SIZE, transform=strong_aug)
val_ds_noaug = BrainSegDataset(DATA_DIR, IMG_SIZE, transform=A.Compose([A.Normalize()]))
train_loader_aug = DataLoader(train_ds_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader_noaug = DataLoader(val_ds_noaug, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
# 3. CNN Improved
cnn_imp = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", classes=NUM_CLASSES)
opt_cnn_imp = optim.AdamW(cnn_imp.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler_cnn = optim.lr_scheduler.CosineAnnealingLR(opt_cnn_imp, T_max=10)
cnn_imp, _, cnn_imp_dice = train_model(cnn_imp, train_loader_aug, val_loader_noaug, epochs=10, lr=3e-4,
                                        loss_fn=loss_fn, optimizer=opt_cnn_imp, scheduler=scheduler_cnn, desc="SMP CNN Imp")

Evaluating: 100%|██████████| 593/593 [01:01<00:00,  9.57it/s]


  Val Dice: 0.2367


Evaluating: 100%|██████████| 593/593 [01:00<00:00,  9.77it/s]


  Val Dice: 0.4549


Evaluating: 100%|██████████| 593/593 [01:01<00:00,  9.68it/s]


...

  Val Dice: 0.6342


Evaluating: 100%|██████████| 593/593 [01:00<00:00,  9.73it/s]


  Val Dice: 0.6276


Evaluating: 100%|██████████| 593/593 [01:06<00:00,  8.88it/s]

  Val Dice: 0.6358


In [10]:
# 4. ViT Improved
vit_imp = smp.Unet(encoder_name="mit_b0", encoder_weights="imagenet", classes=NUM_CLASSES)
opt_vit_imp = optim.AdamW(vit_imp.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_vit = optim.lr_scheduler.CosineAnnealingLR(opt_vit_imp, T_max=10)
loss_fn = nn.CrossEntropyLoss()
vit_imp, _, vit_imp_dice = train_model(vit_imp, train_loader_aug, val_loader_noaug, epochs=10, lr=1e-4,
                                        loss_fn=loss_fn, optimizer=opt_vit_imp, scheduler=scheduler_vit, desc="SMP ViT Imp")

Evaluating: 100%|██████████| 593/593 [01:00<00:00,  9.83it/s]


  Val Dice: 0.0186


Evaluating: 100%|██████████| 593/593 [00:59<00:00,  9.99it/s]


  Val Dice: 0.1121


Evaluating: 100%|██████████| 593/593 [00:59<00:00,  9.98it/s]


...

  Val Dice: 0.6074


Evaluating: 100%|██████████| 593/593 [01:01<00:00,  9.70it/s]


  Val Dice: 0.6145


Evaluating: 100%|██████████| 593/593 [01:01<00:00,  9.68it/s]

  Val Dice: 0.6166


|Модель|Энкодер|Final Dice|Best Dice|
|-|-|-|-|
|SMP CNN Base|ResNet34|0.504|0.562 |
|SMP ViT Base|MiT-B0|0.620|0.620 |
|SMP CNN Improved|ResNet34|0.636|0.636 |
|SMP ViT Improved|MiT-B0|0.617|0.617|

Для CNN-архитектуры:

Улучшения сработали: +13.2 п.п. по Dice (0.504 -> 0.636).

Причины успеха:

- Аугментации компенсировали малый размер датасета, снизив переобучение.

- AdamW + weight decay + CosineAnnealingLR обеспечили более плавную сходимость.

Для свёрточных сетей в медицинской сегментации регуляризация через данные и оптимизатор критически важна.


Для ViT-архитектуры:

- Улучшения не дали прироста - результат остался на уровне ~0.62.

Для трансформерных энкодеров базовой конфигурации может быть достаточно; дальнейший рост требует изменения архитектуры или увеличения данных. Дальнейшие улучшения потребуют увеличения количества эпох.

# Кастомные модели

In [23]:
# Кастомный U-Net (CNN)
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class CustomUNet(nn.Module):
    def __init__(self, num_classes=4, base_channels=32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # ENCODER
        self.enc1 = ConvBlock(3, base_channels)
        self.enc2 = ConvBlock(base_channels, base_channels*2)
        self.enc3 = ConvBlock(base_channels*2, base_channels*4)
        self.enc4 = ConvBlock(base_channels*4, base_channels*8)

        self.bottleneck = ConvBlock(base_channels*8, base_channels*8)

        # DECODER
        self.up4 = nn.ConvTranspose2d(base_channels*8, base_channels*4, 2, stride=2)
        self.dec4 = ConvBlock(base_channels*4 + base_channels*8, base_channels*4)

        self.up3 = nn.ConvTranspose2d(base_channels*4, base_channels*2, 2, stride=2)
        self.dec3 = ConvBlock(base_channels*2 + base_channels*4, base_channels*2)

        self.up2 = nn.ConvTranspose2d(base_channels*2, base_channels, 2, stride=2)
        self.dec2 = ConvBlock(base_channels + base_channels*2, base_channels)

        self.up1 = nn.ConvTranspose2d(base_channels, base_channels, 2, stride=2)
        self.dec1 = ConvBlock(base_channels + base_channels, base_channels)

        self.out_conv = nn.Conv2d(base_channels, num_classes, 1)

    def forward(self, x):
        # Encoder с сохранением skip-connections
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        # Decoder с конкатенацией
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out_conv(d1)

# Кастомный Трансформер-сегментатор (Упрощенный ViT + Decoder)
class SimpleViTSeg(nn.Module):
    def __init__(self, num_classes=4, img_size=256, patch_size=16, embed_dim=128, depth=4, num_heads=4):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size  # 256//16 = 16
        self.num_patches = self.grid_size ** 2    # 256

        # Patch embedding: разбиваем изображение на патчи
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)

        # Позиционные эмбеддинги для патчей
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, embed_dim) * 0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True,
            dim_feedforward=embed_dim*4,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # Convolutional decoder для апсемплинга до исходного разрешения
        self.decoder = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim//2, 3, padding=1),
            nn.BatchNorm2d(embed_dim//2),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(embed_dim//2, embed_dim//4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(embed_dim//4),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(embed_dim//4, embed_dim//8, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(embed_dim//8),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(embed_dim//8, embed_dim//8, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(embed_dim//8),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(embed_dim//8, num_classes, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, x):
        B, _, H, W = x.shape

        patches = self.patch_embed(x)
        B, C, Gh, Gw = patches.shape

        patches_flat = patches.flatten(2).transpose(1, 2)

        # Добавляем позиционные эмбеддинги и пропускаем через трансформер
        tokens = patches_flat + self.pos_embed
        out = self.transformer(tokens)

        # Reshape обратно в пространственную сетку
        out = out.transpose(1, 2).reshape(B, C, Gh, Gw)

        return self.decoder(out)

custom_cnn = CustomUNet(NUM_CLASSES)
custom_vit = SimpleViTSeg(NUM_CLASSES)

# Обучение кастомных моделей (Базовое и Улучшенное)

In [20]:
custom_cnn = CustomUNet(NUM_CLASSES).to(device)
custom_vit = SimpleViTSeg(NUM_CLASSES).to(device)

# Снижаем batch size для кастомных моделей
CUSTOM_BATCH_SIZE = 2
train_loader_custom = DataLoader(train_ds, batch_size=CUSTOM_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader_custom = DataLoader(val_ds, batch_size=CUSTOM_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

train_loader_aug_custom = DataLoader(train_ds_aug, batch_size=CUSTOM_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader_noaug_custom = DataLoader(val_ds_noaug, batch_size=CUSTOM_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [21]:
# 5. Custom CNN Base
loss_fn = nn.CrossEntropyLoss()
opt_ccnn = optim.Adam(custom_cnn.parameters(), lr=1e-3)
custom_cnn, _, ccnn_base_dice = train_model(
    custom_cnn, train_loader_custom, val_loader_custom,
    epochs=10, lr=1e-3, loss_fn=loss_fn, optimizer=opt_ccnn,
    desc="Custom CNN Base"
)

Evaluating: 100%|██████████| 474/474 [00:14<00:00, 33.81it/s]


  Val Dice: 0.0159


Evaluating: 100%|██████████| 474/474 [00:13<00:00, 35.16it/s]


  Val Dice: 0.0665


Evaluating: 100%|██████████| 474/474 [00:12<00:00, 37.44it/s]


...

  Val Dice: 0.0594


Evaluating: 100%|██████████| 474/474 [00:13<00:00, 35.69it/s]


  Val Dice: 0.1247


Evaluating: 100%|██████████| 474/474 [00:13<00:00, 34.42it/s]

  Val Dice: 0.0695


In [24]:
# 6. Custom ViT Base
custom_vit = SimpleViTSeg(
    num_classes=NUM_CLASSES,
    img_size=IMG_SIZE[0],
    patch_size=16,
    embed_dim=64,
    depth=2,
    num_heads=4
).to(device)

opt_cvit = optim.Adam(custom_vit.parameters(), lr=1e-4)
custom_vit, _, cvit_base_dice = train_model(
    custom_vit, train_loader_custom, val_loader_custom,
    epochs=10, lr=1e-4, loss_fn=loss_fn, optimizer=opt_cvit,
    desc="Custom ViT Base"
)

Evaluating: 100%|██████████| 474/474 [00:12<00:00, 39.04it/s]


  Val Dice: 0.0000


Evaluating: 100%|██████████| 474/474 [00:13<00:00, 36.12it/s]


  Val Dice: 0.0000


Evaluating: 100%|██████████| 474/474 [00:12<00:00, 39.39it/s]


...

  Val Dice: 0.0689


Evaluating: 100%|██████████| 474/474 [00:12<00:00, 36.52it/s]


  Val Dice: 0.1775


Evaluating: 100%|██████████| 474/474 [00:12<00:00, 37.36it/s]

  Val Dice: 0.1002


Кастомные модели, обученные с нуля, показали существенно более низкое качество сегментации по сравнению с библиотечными аналогами. Это обусловлено отсутствием предобученных весов, ограниченным числом эпох (10) и упрощённой архитектурой. Динамика функции потерь нестабильна, наблюдаются признаки недообучения.

| Модель | Тип | Final Dice | Best Dice | Отставание от SMP |
|---|---|---|---|---|
| SMP CNN Base | Библиотечная | 0.504 | 0.562 | — |
| SMP ViT Base | Библиотечная | 0.620 | 0.620 | — |
| Custom CNN Base | Имплементированная | 0.070 | 0.125 | в 7 раз ниже |
| Custom ViT Base | Имплементированная | 0.100 | 0.178 | в 6 раз ниже |



In [25]:
# 7. Custom CNN Improved
opt_ccnn_imp = optim.AdamW(custom_cnn.parameters(), lr=3e-4, weight_decay=1e-4)
sched_ccnn = optim.lr_scheduler.CosineAnnealingLR(opt_ccnn_imp, T_max=10)
custom_cnn, _, ccnn_imp_dice = train_model(custom_cnn, train_loader_aug_custom, val_loader_noaug_custom, epochs=10, lr=3e-4,
                                           loss_fn=loss_fn, optimizer=opt_ccnn_imp, scheduler=sched_ccnn, desc="Custom CNN Imp")

Evaluating: 100%|██████████| 2369/2369 [01:07<00:00, 35.22it/s]


  Val Dice: 0.0569


Evaluating: 100%|██████████| 2369/2369 [01:06<00:00, 35.82it/s]


  Val Dice: 0.0939


Evaluating: 100%|██████████| 2369/2369 [01:06<00:00, 35.75it/s]


...

  Val Dice: 0.1345


Evaluating: 100%|██████████| 2369/2369 [01:05<00:00, 36.10it/s]


  Val Dice: 0.1793


Evaluating: 100%|██████████| 2369/2369 [01:03<00:00, 37.32it/s]

  Val Dice: 0.1219


In [26]:
# 8. Custom ViT Improved
custom_vit_imp = SimpleViTSeg(
    num_classes=NUM_CLASSES,
    img_size=IMG_SIZE[0],
    patch_size=16,
    embed_dim=64,
    depth=2,
    num_heads=4
).to(device)

opt_cvit_imp = optim.AdamW(custom_vit_imp.parameters(), lr=1e-4, weight_decay=1e-4)
sched_cvit = optim.lr_scheduler.CosineAnnealingLR(opt_cvit_imp, T_max=10)
custom_vit_imp, _, cvit_imp_dice = train_model(
    custom_vit_imp, train_loader_aug_custom, val_loader_noaug_custom,
    epochs=10, lr=1e-4, loss_fn=loss_fn, optimizer=opt_cvit_imp,
    scheduler=sched_cvit, desc="Custom ViT Imp"
)

Evaluating: 100%|██████████| 2369/2369 [01:00<00:00, 38.87it/s]


  Val Dice: 0.0000


Evaluating: 100%|██████████| 2369/2369 [01:02<00:00, 37.71it/s]


  Val Dice: 0.0001


Evaluating: 100%|██████████| 2369/2369 [01:00<00:00, 39.21it/s]


...

  Val Dice: 0.1124


Evaluating: 100%|██████████| 2369/2369 [01:00<00:00, 39.13it/s]


  Val Dice: 0.1025


Evaluating: 100%|██████████| 2369/2369 [01:00<00:00, 38.85it/s]

  Val Dice: 0.1061


Применение техник улучшенного пайплайна к кастомным моделям дало умеренный прирост относительно их базовых версий, однако качество остаётся существенно ниже библиотечных аналогов. Основная причина — отсутствие предобученных весов и ограниченное число эпох (10), что недостаточно для сходимости сетей, обучаемых «с нуля» на малом медицинском датасете.

| Модель | Тип | Final Dice | Best Dice | Отставание от SMP Imp |
|---|---|---|---|---|
| SMP CNN Imp | Библиотечная (улучш.) | 0.636 | 0.636 | — |
| SMP ViT Imp | Библиотечная (улучш.) | 0.617 | 0.617 | — |
| Custom CNN Imp | Кастомная (улучш.) | 0.122 | 0.179 | в 5.2 раз ниже |
| Custom ViT Imp | Кастомная (улучш.) | 0.106 | 0.112 | в 5.8 раз ниже |

Выводы

- Улучшения работают, но ограничены: аугментации и AdamW+CosineLR повысили Dice кастомного CNN на +74%, а ViT на +6% относительно их базовых версий, подтверждая универсальность данных техник.
- Важна роль предобучения: библиотечные модели с ImageNet-весами превосходят кастомные в 5–6 раз. Без трансферного обучения 10 эпох недостаточно для извлечения релевантных медицинских признаков.
- Архитектурные особенности: кастомный CNN реагирует на улучшения лучше, чем ViT, благодаря меньшему числу параметров и более простой оптимизации. Гибридный ViT требует больше данных/эпох для выхода из зоны «случайного угадывания».
- Практическая рекомендация: для достижения качества >0.60 на данном датасете необходимо использовать предобученные энкодеры или увеличить объём обучения до 50–100 эпох с сохранением улучшенного пайплайна.